In [1]:
import requests
import math
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
import json

load_dotenv()

KEY = os.getenv('API_KEY')
LIST_URL = "https://play.limitlesstcg.com/api/tournaments/"
URL = LIST_URL+"{}"
FILENAME = "data.xlsx"

# Tournaments List

In [2]:
start_date = datetime(2025, 3, 27).date()
end_date = datetime(2025, 3, 30).date()

In [3]:
response = requests.get(LIST_URL, headers={'X-Access-Key':KEY}, params={'game': 'PTCG', 'format': 'STANDARD', 'limit': 1000})

In [4]:
tournament_list = []
id_list = []
date_format = "%Y-%m-%dT%H:%M:%S.%fZ"
i = 0
for entry in response.json():
    # print(entry)
    date = (datetime.strptime(entry['date'], date_format) - timedelta(hours=5)).date()
    if entry['players'] >= 64 and date >= start_date and date <= end_date:
        print(entry['id'], entry['name'] )
        tournament_list.append('{}_{}'.format(i,entry['name']))
        id_list.append(entry['id'])
        i+=1


67ce44290bff7e07c136c935 Ditto Together #1 JTG is Legal! (200 Codes)
67e9727a142155bb4448f867 Poli's Pop Up Mickey! #21 ATLANTA TESTING
67e07e41fb3a86a4cffce220 Pokémon Battle Park-Post Rotation Tournament
67bfd40078c0b323188217d6 Redacted 2nd Chance #10 (Mini Event 50 Codes)
67bfd08678c0b323188217bb Redacted Evolutions #10 (100 Codes) CLOSED LISTS
677dade77960ec710efe5294 ToD: "Regional" Championships V
67e19be0fb3a86a4cffcebab Special Battles #1
67e18a64fb3a86a4cffceafd Redacted x Ditto x PokeArena Cup (600 Codes+Cash)
67a913fd05f4a8b0fc5726b9 Polskie Nocne Granie #4
67e07d6cfb3a86a4cffce215 Pokémon Battle Park- Journey Together is Legal
679e9aa2ac375d44ca9134d8 🐍TLS TCG SERIES QUALIFIER 8/10 S2 - CASH + CODES
67de1d1440ab762997ea9d10 🏆Moujii's Dojo🏆Journey Together Legal🏆


# Tournament Data

In [5]:
deck_df = pd.DataFrame(columns=['Player', 'Nation', 'Deck', 'Tournament', 'Placement', 'Day2'])
# standing_df = pd.DataFrame(columns=['Player', 'Wins', 'Losses', 'Ties'])
# pairings_df = pd.DataFrame(columns=['Tour', 'Round', 'Player', 'Opponent', 'Result'])
matchups_df = pd.DataFrame(columns=['Deck', 'Opposing Deck', 'Wins', 'Losses', 'Ties'])

with open('archetype.json', 'r') as file:
    arch_dict = json.load(file)
    # deck_df['Deck'] = deck_df['Deck'].apply(lambda x: arch_dict[x] if x in arch_dict else x)

deck_dict = deck_df.set_index('Player')['Deck'].to_dict()

In [6]:
FILENAME = "data.xlsx"
# standings = requests.get(URL.format(id_list[0])+"/standings", headers={'X-Access-Key':KEY})

deck_df = pd.DataFrame(columns=['Player', 'Nation', 'Deck', 'Tournament', 'Placement', 'Day2'])
# standing_df = pd.DataFrame(columns=['Player', 'Wins', 'Losses', 'Ties'])
# pairings_df = pd.DataFrame(columns=['Tour', 'Round', 'Player', 'Opponent', 'Result'])
matchups_df = pd.DataFrame(columns=['Deck', 'Opposing Deck', 'Wins', 'Losses', 'Ties'])

with open('archetype.json', 'r') as file:
    arch_dict = json.load(file)
    deck_df['Deck'] = deck_df['Deck'].apply(lambda x: arch_dict[x] if x in arch_dict else x)

deck_dict = deck_df.set_index('Player')['Deck'].to_dict()

ignore_list = []

for id, tour in zip(id_list, tournament_list):
    if id in ignore_list:
        continue
    standings = requests.get(URL.format(id)+"/standings", headers={'X-Access-Key':KEY})
    pairings = requests.get(URL.format(id)+"/pairings", headers={'X-Access-Key':KEY})
    phases = requests.get(URL.format(id)+"/details", headers={'X-Access-Key':KEY}).json()['phases']
    if not all('name' in entry['deck'] for entry in standings.json()):
        print("SKIPPED:", id, tour)
        continue

    # print(tour)
    top_size = 0
    swiss_rounds = 0
    if len(phases) <= 1:
        top_size = 8
    else:
        for phase in phases:
            if phase['type'] != 'SWISS': break
            swiss_rounds += phase['rounds']
    # i = 1
    for entry in standings.json():

        # if entry['placing'] == 'None': continue
        name = "{}_{}".format(tour, entry['player'])
        nation = entry['country']
        wins = entry['record']['wins']
        losses = entry['record']['losses']
        ties = entry['record']['ties']
        deck = arch_dict[entry['deck']['name']] if entry['deck']['name'] in arch_dict else entry['deck']['name']
        # deck_df.loc[len(deck_df)] = name, entry['deck']['name']

        if top_size > 0:
            if entry['placing'] != None and entry['placing'] < (top_size+1):
                deck_df.loc[len(deck_df)] = name, nation, deck, tour, "Top {}".format(pow(2, math.ceil(math.log(entry['placing'], 2)))), True
            else:
                deck_df.loc[len(deck_df)] = name, nation, deck, tour, "Out of Top", False
        else:
            if entry['placing'] != None and (wins + losses + ties) > swiss_rounds:
                deck_df.loc[len(deck_df)] = name, nation, deck, tour, "Top {}".format(pow(2, math.ceil(math.log(entry['placing'], 2)))), True
            else:
                deck_df.loc[len(deck_df)] = name, nation, deck, tour, "Out of Top", False

    for entry in pairings.json():
        # if entry['round'] < 4:
        #     continue
        try:
            player = "{}_{}".format(tour, entry['player1'])
            opponent = "{}_{}".format(tour, entry['player2'])
            player_deck = deck_df.loc[deck_df['Player'] == player]['Deck'].values[0]
            opp_deck = deck_df.loc[deck_df['Player'] == opponent]['Deck'].values[0]
            matchup = matchups_df.loc[(matchups_df['Deck'] == player_deck) & (matchups_df['Opposing Deck'] == opp_deck)]
            inv_matchup = matchups_df.loc[(matchups_df['Deck'] == opp_deck) & (matchups_df['Opposing Deck'] == player_deck)]
            if entry['winner'] == 0:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'T'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'T'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 0, 1
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 0, 0, 1
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 0, 2
                else:
                    matchups_df.loc[matchup.index, 'Ties'] += 1
                    matchups_df.loc[inv_matchup.index, 'Ties'] += 1
            elif entry['player1'] == entry['winner']:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'W'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'L'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 0, 0
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 0, 1, 0
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 1, 0
                else:
                    matchups_df.loc[matchup.index, 'Wins'] += 1
                    matchups_df.loc[inv_matchup.index, 'Losses'] += 1
            elif entry['player2'] == entry['winner']:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'L'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'W'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 1, 0
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 1, 0, 0
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 1, 0
                else:
                    matchups_df.loc[matchup.index, 'Losses'] += 1
                    matchups_df.loc[inv_matchup.index, 'Wins'] += 1
        except:
            continue




SKIPPED: 67de1d1440ab762997ea9d10 11_🏆Moujii's Dojo🏆Journey Together Legal🏆


In [7]:
with pd.ExcelWriter(FILENAME) as writer:
    deck_df.to_excel(writer, sheet_name='decks', index=False)
    # standing_df.to_excel(writer, sheet_name='standings', index=False)
    # pairings_df.to_excel(writer, sheet_name='pairings', index=False)
    matchups_df.to_excel(writer, sheet_name='matchups', index=False)